# Chapter 6: Evaluating GenAI Applications with MLflow

## 📓 About this notebook
This notebook evaluates the Unity Airways agent with MLflow's GenAI evaluation harness: building evaluation datasets, defining code-based and LLM-judge scorers, running and comparing evaluations, simulating multi-turn conversations, and capturing human feedback.

**Maps to the book:** Chapter 6, *Evaluating GenAI Applications with MLflow* — sections: The MLflow Evaluation Framework (Evaluation Harness, Evaluation Modes); Creating and Managing Evaluation Datasets; Designing and Managing Scorers (Code-Based Scorers, LLM Judges); Running and Comparing Evaluations in MLflow; Capturing Human Feedback.

### ✅ Prerequisites

Run these before this notebook, or the evaluation cells will fail:

1. [`Appendix/data_ingestion`](../Appendix/data_ingestion) — loads the Unity Airways datasets (this notebook reads the `qa_dataset` table) and builds the FAQ vector search index.
2. [Chapter 4](../Chapter04) — registers the `tool_calling_agent` model in Unity Catalog, which this notebook loads and evaluates.

Catalog and schema come from [`conf/data.yml`](../conf/data.yml). See the [repository README](../README.md) for full setup.

In [0]:
%pip install -r ../requirements.txt
dbutils.library.restartPython()

In [ ]:
# Set the Unity Catalog and schema for Unity Airways data and models
# These variables are used to reference tables and models throughout the notebook
import yaml
with open('../conf/data.yml') as f:
    default_uc = yaml.safe_load(f)['default_uc']
CATALOG = default_uc['catalog']
SCHEMA = default_uc['schema']

## Setup and cleanup
Initialize the environment and clean up datasets, scorers, and labeling sessions from earlier runs so the walkthrough starts from a known state. _(see Ch 6, "The MLflow Evaluation Framework")_

In [0]:
from mlflow.genai.datasets import delete_dataset

# Delete an existing dataset
delete_dataset(
        name=f"{CATALOG}.{SCHEMA}.eval_dataset"
    )
print("Deleted the old dataset.")

In [0]:
from mlflow.genai.scorers import delete_scorer

# Delete any existing undeserializable scorers by name
for name in ["relevance_to_query", "safety", "retrieval_groundedness", "correctness",
             "professional_tone", "response_completeness", "coherence", "response_length"]:
    try:
        delete_scorer(name=name)
    except Exception:
        pass

In [ ]:
import mlflow
import mlflow.genai.labeling as labeling
import mlflow.genai.label_schemas as schemas

# Set a workspace experiment (numeric ID) for labeling sessions
mlflow.set_experiment(f"/Users/YOUR_EMAIL_ADDRESS/trace_review_exp")

# Create the labeling session – it will live in the active experiment
session = labeling.create_labeling_session(
    name="trace_review_exp",
    assigned_users=["REVIEWER_EMAIL_ADDRESS"], # The email addresses in the assigned_users list must correspond to valid users in the Databricks workspace.
    label_schemas=[schemas.EXPECTED_RESPONSE],
)

print("Experiment for this session:", session.experiment_id)
print("Labeling session URL:", session.url)

In [0]:
import mlflow
from mlflow.genai.datasets import create_dataset

# Create a managed evaluation dataset
first_dataset = create_dataset(
    name=f"{CATALOG}.{SCHEMA}.my_first_dataset",
)
print(f"Created evaluation dataset: {first_dataset.name}")

In [ ]:
EXPERIMENT_IDS = ["<your_experiment_id>"]

# Retrieve traces from previous experiment
traces = mlflow.search_traces(
    experiment_ids=EXPERIMENT_IDS,
    filter_string=("traces.status = 'OK'"),
    order_by=["attributes.timestamp_ms DESC"],
    max_results=10
)
print(f"Found {len(traces)} successful traces")

# Transform traces into the format expected by merge_records
records = [
    {
        "inputs": {"question": row["request"]["messages"][0]["content"]},
    }
    for _, row in traces.iterrows()
]

# Add the traces to the evaluation dataset
first_dataset = first_dataset.merge_records(records)
print(f"Added {len(records)} records to evaluation dataset")

In [0]:
# Using domain expert labels
import mlflow.genai.labeling as labeling

# Retrieve available labeling sessions
sessions = labeling.get_labeling_sessions()
for s in sessions:
    print(f"Session: {s.name}  (ID: {s.labeling_session_id})")

# Sync the first labeling session into the evaluation dataset
sessions[0].sync(to_dataset=f"{CATALOG}.{SCHEMA}.my_first_dataset")
print("Synchronized expert labels into evaluation dataset.")

In [0]:
# Using curated examples

curated_examples = [
    {
        "inputs": {"question": "Can I refund a partially used ticket for a delayed flight?"},
        "expectations": {"expected_response": "Refund depends on fare type and delay duration."}
    },
    {
        "inputs": {"question": "Can I change my Lite ticket for tomorrow?"},
        "expectations": {"expected_response": "Lite tickets cannot be changed after 24 hours of purchase."}
    },
    {
        "inputs": {"question": "Refund for a cancelled flight with waiver code WX-2025?"},
        "expectations": {"expected_response": "Refund permitted if the waiver code is valid for the affected segment."}
    }
]
first_dataset = first_dataset.merge_records(curated_examples)

In [0]:
# Read 50 samples from Unity Airways QA dataset in Unity Catalog
eval_data_df = spark.read.table(
    f"{CATALOG}.{SCHEMA}.qa_dataset"
).limit(50)

display(eval_data_df)

In [0]:
# Create an evaluation dataset
eval_dataset_new = create_dataset(
    name=f"{CATALOG}.{SCHEMA}.eval_dataset",
)
print(f"Created evaluation dataset: {eval_dataset_new.name}")

# Transform columns to match evaluation dataset schema
imported_data = eval_data_df.selectExpr(
    "struct(Question as question) as inputs",
    "struct(Answer as expected_response) as expectations"
)
# Update the dataset with imported cases
eval_dataset_new = eval_dataset_new.merge_records(imported_data)

In [0]:
# Merge imported data with my_first_dataset
first_dataset = first_dataset.merge_records(imported_data)

In [0]:
from mlflow.genai.datasets import get_dataset

# Load an existing dataset
eval_dataset_imported = get_dataset(
        name=f"{CATALOG}.{SCHEMA}.eval_dataset"
    )

eval_dataset_imported = eval_dataset_imported.merge_records(curated_examples)
print("Updated the imported dataset with curated examples.")


In [0]:
from mlflow.genai.datasets import delete_dataset

# Delete an existing dataset
delete_dataset(
        name=f"{CATALOG}.{SCHEMA}.my_first_dataset"
    )

print("Deleted the old dataset.")

## Scorers

In [0]:
from mlflow.genai.scorers import scorer
from mlflow.entities import Feedback

@scorer(name="response_length")
def ua_response_length_scorer(outputs) -> Feedback:
    response = outputs if isinstance(outputs, str) else outputs.get('response') or outputs.get('outputs', {}).get('response', '')
    word_count = len(str(response).split())

    if word_count < 5:
        return Feedback(value=0.0, rationale=f"Response too short ({word_count} words)")
    elif word_count > 120:
        return Feedback(value=0.5, rationale=f"Response quite long ({word_count} words)")
    else:
        return Feedback(value=1.0, rationale=f"Appropriate length ({word_count} words)")

In [0]:
from mlflow.genai.scorers import (
    RelevanceToQuery,
    Safety,
    RetrievalGroundedness,
    Correctness,
)

In [0]:
eval_model = "databricks:/databricks-gpt-oss-120b"
ua_relevancy_scorer = RelevanceToQuery(model=eval_model)
ua_safety_scorer = Safety(model=eval_model)
ua_groundedness_scorer = RetrievalGroundedness(model=eval_model)
ua_correctness_scorer = Correctness(model=eval_model)

## Designing and managing scorers
Define a code-based length scorer, configure built-in LLM judges, and create custom judges (tone, completeness, coherence) for comprehensive evaluation.

In [0]:
from mlflow.genai.scorers import Guidelines

ua_professional_tone_scorer = Guidelines(
    name="professional_tone",
    guidelines="""
    The response must use professional, courteous language appropriate for airline customer service.
    Requirements:
    - Use polite and respectful language
    - Avoid casual expressions or slang
    - Maintain a helpful and solution-oriented tone
    - Include appropriate greetings or closings when relevant
    """,
    model=eval_model,
)

ua_completeness_scorer = Guidelines(
    name="response_completeness", 
    guidelines="""
    The response must completely address the customer's question:
    - Directly answer the specific question asked
    - Provide all relevant details mentioned in the expected response
    - Include next steps or additional resources when appropriate
    - Avoid generic responses when specific information is requested
    """,
    model=eval_model,
)

In [0]:
from mlflow.genai.judges import make_judge
from typing import Literal

ua_coherence_scorer = make_judge(
    name="coherence",
    instructions=(
       "Evaluate if the response is coherent, maintaining a constant tone, and following airline policies"
       "Question: {{ inputs }}\n"
       "Response: {{ outputs }}\n"
       "Categorize the response as 'coherent', 'somewhat coherent', or 'incoherent'."
   ),
   #feedback_value_type=Literal["coherent", "somewhat coherent", "incoherent"],
    #model=eval_model,
)

In [0]:
# Register built-in LLM judges
ua_relevancy_scorer.register()
ua_safety_scorer.register()
ua_groundedness_scorer.register()
ua_correctness_scorer.register()

# Register custom LLM judges
ua_professional_tone_scorer.register()
ua_completeness_scorer.register()
ua_coherence_scorer.register()

# Register code-based scorers
ua_response_length_scorer.register()

## Evaluation

In [0]:
MODEL_VERSION = 1

In [0]:
import mlflow

# Define version and model location
model_uri = f"models:/{CATALOG}.{SCHEMA}.tool_calling_agent/{MODEL_VERSION}"

# Load the LangChain model from Unity Catalog and test
loaded_agent = mlflow.langchain.load_model(model_uri)
response = loaded_agent.invoke({'messages': [{'content': 'How do i book flights with Unity Airways?',
   'role': 'user'}]})
print(response)

## Running an evaluation

In [0]:
from mlflow.genai.datasets import get_dataset

# Load the evaluation dataset
eval_dataset = get_dataset(
    name=f"{CATALOG}.{SCHEMA}.eval_dataset"
)

In [0]:
from mlflow.genai.scorers import get_scorer

# Load scorers
ua_relevancy_scorer = get_scorer(name="relevance_to_query")
ua_safety_scorer = get_scorer(name="safety")
ua_groundedness_scorer = get_scorer(name="retrieval_groundedness")
ua_correctness_scorer = get_scorer(name="correctness")
ua_professional_tone_scorer = get_scorer(name="professional_tone")
ua_completeness_scorer = get_scorer(name="response_completeness")
ua_coherence_scorer = get_scorer(name="coherence")
ua_response_length_scorer = get_scorer(name="response_length")

# Assemble scorers into a list
unity_airways_scorers = [
    ua_relevancy_scorer,
    ua_safety_scorer,
    ua_groundedness_scorer,
    ua_correctness_scorer,
    ua_professional_tone_scorer,
    ua_completeness_scorer,
    ua_coherence_scorer,
    ua_response_length_scorer
]

In [0]:
from typing import Dict

# Wraps the Unity Airways agent so that mlflow.genai.evaluate can call it.
def agent_predict_fn(question: str) -> Dict[str, str]:
    return loaded_agent.invoke({
        "messages": [
            {"role": "user", "content": question}
        ]
    })

In [ ]:
import mlflow

# Run evaluation with predefined scorers
with mlflow.start_run(run_name=f"ua_rag_eval_baseline"):
    mlflow.set_tag("model_version", MODEL_VERSION)
    mlflow.set_tag("dataset_version", "eval_dataset")
    eval_results = mlflow.genai.evaluate(
        data=eval_dataset,
        predict_fn=agent_predict_fn,
        scorers=unity_airways_scorers
    )

## Multiturn Evaluation

In [ ]:
# Shared session/conversation ID used to group turns for multi-turn evaluation
session_id = "unity-airways-multiturn-session"

mlflow.update_current_trace(metadata={"mlflow.trace.session": session_id})

In [0]:
import mlflow
from mlflow.genai.simulators import ConversationSimulator

# Unity Air Test Cases
test_cases = [
    {
        "goal": "Reschedule booking id xaji0y6dpbhsahxt, confirm the fare difference and whether the LITE fare brand allows changes, and handle the expected rejection gracefully",
        "persona": "A frequent business traveler who is direct and impatient, expects quick answers, and pushes back if the agent is vague about costs or policies.",                                                                                           
    },
    {
      "goal": "Ask about the refund status for booking id xxc02dgtfgq5c34d. Verify that the refund has been processed",
      "persona": "A non-technical traveler who speaks in casual language, sometimes provides incomplete information, and needs the agent to ask clarifying questions."
    },
]

# Conversation Simulator
simulator = ConversationSimulator(
    test_cases=test_cases,
    max_turns=5,
)


In [0]:
from mlflow.genai.judges import make_judge                                                                                    
from typing import Literal                                                                                                    
                                                                                                                                
professional_tone_judge = make_judge(
    name="professional_tone",                                                                                                 
    instructions=(
        "Review this conversation:\n\n{{ conversation }}\n\n"
        "Evaluate whether the assistant maintained a professional, courteous, "
        "and airline-appropriate tone throughout the conversation. "
        "Consider: empathy when handling complaints, clear and polite language, "
        "and avoiding overly casual or robotic responses.\n\n"
        "Rate as true if professional, false otherwise."
    ),
    feedback_value_type=bool,
)


In [ ]:
from mlflow.genai.scorers import ConversationCompleteness

results = mlflow.genai.evaluate(
    data=simulator,
    predict_fn=agent_predict_fn,
    scorers=[
        ConversationCompleteness(),
        Safety(), #this is a single-turn judge
        professional_tone_judge
    ],
)


## Capturing Human Feedback
Log end-user thumbs up/down and developer scores against traces, and set up label schemas and labeling sessions for domain-expert review.

In [0]:
# Unity Airways: capturing end-user feedback
import mlflow
from typing import Optional
from mlflow.entities.assessment import AssessmentSource, AssessmentSourceType
def log_end_user_feedback(
    trace_id: str,
    satisfied: bool,
    rationale: Optional[str] = None,
    user_id: Optional[str] = None,
) -> dict:
    """
    Record a thumbs-up or thumbs-down from the chat UI against a specific trace.
    - trace_id: the MLflow trace identifier returned with the chat response
    - satisfied: True for thumbs-up, False for thumbs-down
    - rationale: optional short comment or category label
    - user_id: optional application user identifier
    """
    mlflow.log_feedback(
        trace_id=trace_id,
        name="user_feedback",
        value=satisfied,
        rationale=rationale,
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id=user_id,
        ),
    )
    return {"status": "success", "trace_id": trace_id}

In [0]:
import mlflow
from typing import Optional
from mlflow.entities.assessment import AssessmentSource, AssessmentSourceType

def mark_dev_accuracy(trace_id: str, score: float, note: Optional[str], dev_email: str) -> None:
    mlflow.log_feedback(
        trace_id=trace_id,
        name="dev_accuracy_score",
        value=score,
        rationale=note,
        source=AssessmentSource(
            source_type=AssessmentSourceType.HUMAN,
            source_id=dev_email,
        ),
    )

In [0]:
import mlflow
from mlflow.genai.label_schemas import create_label_schema, InputCategorical, InputText
from mlflow.genai.labeling import create_labeling_session

# Define expert labels
accuracy_schema = create_label_schema(
    name="response_accuracy",
    type="feedback",
    title="Is the response factually accurate?",
    input=InputCategorical(options=["Accurate", "Partially Accurate", "Inaccurate"]),
    overwrite=True,
)

ideal_schema = create_label_schema(
    name="expected_response",
    type="expectation",
    title="What would be the ideal response?",
    input=InputText(),
    overwrite=True,
)

# Create a labeling session and add candidate traces
session = create_labeling_session(
    name="refund_policy_review",
    label_schemas=[accuracy_schema.name, ideal_schema.name],
)

# Select traces to review. Example: the most recent 20 with a thumbs-down.
traces = mlflow.search_traces(
    filter_string="assessments.user_feedback = false",
    max_results=20
)
session.add_traces(traces)
print("Share with experts:", session.url)  # Review link for domain experts